# Janus 00B｜大盤／產業資料獨立補收 v3.8.4（Colab）

不需要執行個股 Notebook；不修改個股 canonical/CURRENT.json，也不貼標。初次預設只探查單月（最多 4 請求），確認來源真實 JSON schema 及授權後再切完整補收。來源回傳 404 或 HTML 時，同次執行停止該來源，避免重複浪費請求。


## 1．環境與 Drive


In [ ]:
import sys, subprocess
from pathlib import Path
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'requests==2.32.5', 'pandas==2.2.3', 'pyarrow==20.0.0'])
from google.colab import drive
drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/個人用/janusChatGPT')
ROOT = PROJECT/'資料源'/'JanusDataset_v3_202610'
if not (ROOT/'data').is_dir():
    raise FileNotFoundError('找不到已存在的資料集：'+str(ROOT))
print('已找到資料集，只寫入衍生研究分區：', ROOT)


## 2．執行參數


In [ ]:
# 初次先做單月探查。檢視報告與原始回應後才把 PROBE_ONLY 改為 False。
PROBE_ONLY = True
START_MONTH = '2025-01'
END_MONTH_OVERRIDE = None
MAX_REQUESTS = 70
MAX_MINUTES = 15
REQUEST_INTERVAL_SECONDS = 2.5
INCLUDE_TWSE_SECTOR = True
INCLUDE_TPEX_SECTOR = True

from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd
last = str(pd.Period(datetime.now(ZoneInfo('Asia/Taipei')).date(), freq='M')-1)
END_MONTH = START_MONTH if PROBE_ONLY else (END_MONTH_OVERRIDE or last)
REQUEST_BUDGET = min(4, MAX_REQUESTS) if PROBE_ONLY else MAX_REQUESTS
print('模式：', '一個月份探查' if PROBE_ONLY else '增量補收', START_MONTH, '~', END_MONTH)


## 3．大盤／產業獨立採集


In [ ]:
# 本格內嵌相同版本的 official_index_backfill.py，不需要 clone GitHub。
BACKFILL_CODE = "\"\"\"Janus v3.8.4 official market/sector history backfill (Colab collection only).\n\nRead-only for canonical/CURRENT.json. Writes immutable research-only source\npartitions under derived/factor_inputs/official_index_backfill/v3.8.4.\nNo ML labels or point-in-time security-industry membership are generated.\n\"\"\"\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport io\nimport json\nimport os\nimport re\nimport time\nfrom dataclasses import dataclass\nfrom datetime import date, datetime, timezone\nfrom pathlib import Path\nfrom zoneinfo import ZoneInfo\nfrom urllib.parse import urlencode\n\nimport requests\n\nVERSION = 'v3.8.4'\nSOURCES = {\n    'twse_market': {\n        'endpoint': 'https://www.twse.com.tw/indicesReport/MI_5MINS_HIST',\n        'source_page': 'https://www.twse.com.tw/zh/indices/taiex/mi-5min-hist.html',\n        'market': 'TWSE', 'kind': 'market_index',\n    },\n    'twse_sector': {\n        'endpoint': 'https://www.twse.com.tw/rwd/zh/afterTrading/MI_INDEX',\n        'source_page': 'https://www.twse.com.tw/rwd/zh/afterTrading/MI_INDEX',\n        'market': 'TWSE', 'kind': 'sector_index',\n    },\n    'tpex_market': {\n        'endpoint': 'https://www.tpex.org.tw/zh-tw/mainboard/trading/info/daily-indices.html',\n        'source_page': 'https://www.tpex.org.tw/zh-tw/mainboard/trading/info/daily-indices.html',\n        'market': 'TPEX', 'kind': 'market_index',\n    },\n    'tpex_sector': {\n        'endpoint': 'https://www.tpex.org.tw/zh-tw/mainboard/trading/statistics/indices/daily-sector.html',\n        'source_page': 'https://www.tpex.org.tw/zh-tw/mainboard/trading/statistics/indices/daily-sector.html',\n        'market': 'TPEX', 'kind': 'sector_index',\n    },\n}\nRIGHTS = {\n    'twse_market': 'https://data.gov.tw/dataset/11755',\n    'tpex_market': 'https://data.gov.tw/dataset/11391',\n    'twse_sector': 'https://www.twse.com.tw/zh/products/information/history.html',\n    'tpex_sector': 'https://www.tpex.org.tw/zh-tw/mainboard/trading/statistics/indices/daily-sector.html',\n}\nSTORAGE_STATUS = 'RESEARCH_DATA_ONLY'\nSUCCEEDED = 'VERIFIED'\nBLOCK_STATUSES = (401, 403, 429)\nMAX_RAW_BYTES = 6_000_000\n\n\nclass SchemaBlocked(ValueError):\n    \"\"\"Official source schema/time value not proven: do not ingest.\"\"\"\n\n\ndef sha_bytes(raw: bytes) -> str:\n    return hashlib.sha256(raw).hexdigest()\n\n\ndef write_new(path: Path, raw: bytes) -> str:\n    \"\"\"Never overwrite validated input/output; allow exact idempotent writes.\"\"\"\n    path.parent.mkdir(parents=True, exist_ok=True)\n    digest = sha_bytes(raw)\n    if path.exists():\n        if sha_bytes(path.read_bytes()) != digest:\n            raise ValueError(f'IMMUTABLE_CONFLICT: {path}')\n        return digest\n    tmp = path.with_name(path.name + '.tmp')\n    try:\n        if tmp.exists():\n            tmp.unlink()\n        tmp.write_bytes(raw)\n        os.replace(tmp, path)\n    finally:\n        if tmp.exists():\n            tmp.unlink()\n    return digest\n\n\ndef json_bytes(data: dict | list) -> bytes:\n    return (json.dumps(data, ensure_ascii=False, sort_keys=True, separators=(',', ':'), default=str)+'\\n').encode('utf-8')\n\n\ndef _date(s) -> str:\n    s = str(s).strip().replace('年','/').replace('月','/').replace('日','').replace(' ', '')\n    if re.fullmatch(r'20\\d{6}', s):\n        return date(int(s[:4]), int(s[4:6]), int(s[6:])).isoformat()\n    m = re.fullmatch(r'(\\d{2,4})[/\\-](\\d{1,2})[/\\-](\\d{1,2})', s)\n    if not m:\n        raise SchemaBlocked(f'UNVERIFIED_DATE: {str(s)[:60]}')\n    yr=int(m[1]); yr = yr+1911 if yr<1911 else yr\n    try:\n        return date(yr,int(m[2]),int(m[3])).isoformat()\n    except ValueError as exc:\n        raise SchemaBlocked(f'INVALID_DATE: {s}') from exc\n\n\ndef _month(v: str) -> str:\n    if not re.fullmatch(r'20\\d{2}-(0[1-9]|1[0-2])', v):\n        raise ValueError('BAD_MONTH_FORMAT')\n    return v\n\n\ndef months_between(first: str, last: str) -> list[str]:\n    first=_month(first);last=_month(last)\n    fy,fm=map(int,first.split('-'));ly,lm=map(int,last.split('-'))\n    if (fy,fm)>(ly,lm):\n        raise ValueError('START_AFTER_END')\n    if (ly-fy)*12+lm-fm>120:\n        raise ValueError('BACKFILL_RANGE_TOO_LARGE')\n    return [f'{y:04d}-{m:02d}' for y in range(fy,ly+1) for m in range(1,13)\n            if (y,m)>=(fy,fm) and (y,m)<=(ly,lm)]\n\n\ndef _number(v) -> float:\n    raw=str(v).strip().replace(',','').replace('＋','+').replace('−','-')\n    if not re.fullmatch(r'\\+?\\d+(?:\\.\\d+)?', raw):\n        raise SchemaBlocked(f'BAD_INDEX_NUMBER: {str(v)[:48]}')\n    n=float(raw)\n    if not 0<n<10**9:\n        raise SchemaBlocked('NONPOSITIVE_OR_IMPLAUSIBLE_INDEX')\n    return n\n\n\ndef _fields(rows: list, cols: list[str], *, title: str = ''):\n    if not isinstance(rows,list) or not isinstance(cols,list) or not rows:\n        raise SchemaBlocked('EMPTY_OR_UNKNOWN_TABLE')\n    if any(not isinstance(row,(list,tuple)) or len(row)!=len(cols) for row in rows):\n        raise SchemaBlocked('ROW_FIELD_LENGTH_MISMATCH')\n    return [{str(k).strip():v for k,v in zip(cols,row)} for row in rows]\n\n\ndef _table_objects(payload: dict):\n    if not isinstance(payload,dict):\n        raise SchemaBlocked('OBJECT_REQUIRED')\n    if payload.get('stat') and str(payload['stat']).upper() not in ('OK','SUCCESS'):\n        raise SchemaBlocked('NON_OK_PROVIDER_STAT')\n    tables = []\n    for t in payload.get('tables', []) if isinstance(payload.get('tables'),list) else []:\n        if not isinstance(t,dict):\n            continue\n        cols=t.get('fields') or t.get('columns') or t.get('head') or t.get('headers')\n        vals=t.get('data') or t.get('rows') or t.get('aaData')\n        if isinstance(cols,list) and isinstance(vals,list):\n            tables.append((str(t.get('title','')), _fields(vals, cols)))\n        elif isinstance(vals,list) and vals and all(isinstance(r,dict) for r in vals):\n            tables.append((str(t.get('title','')), vals))\n    # TWSE often uses fields1/data1, fields2/data2 etc.\n    for key,cols in payload.items():\n        if not re.fullmatch(r'fields\\d*', key):\n            continue\n        suffix=key[6:]\n        vals=payload.get('data'+suffix)\n        if isinstance(cols,list) and isinstance(vals,list) and vals:\n            title=payload.get('title'+suffix,'')\n            tables.append((str(title),_fields(vals,cols)))\n    for key in ('data','rows','aaData'):\n        values=payload.get(key)\n        if isinstance(values,list) and values and all(isinstance(r,dict) for r in values):\n            tables.append((str(payload.get('title','')),values))\n    return tables\n\n\ndef _get_col(row: dict, choices: tuple[str,...], *, contains: bool = False):\n    for expected in choices:\n        for col,val in row.items():\n            name=re.sub(r'\\s+', '',str(col))\n            if name == expected or (contains and expected in name):\n                return val\n    return None\n\n\ndef _dedup(rows: list[dict], keys: tuple[str,...]) -> list[dict]:\n    rows=sorted(rows,key=lambda x:tuple(str(x[k]) for k in keys))\n    sig=[]\n    for row in rows:\n        tag=tuple(row[k] for k in keys)\n        if sig and tag==sig[-1]:\n            raise SchemaBlocked('DUPLICATE_OFFICIAL_KEY')\n        sig.append(tag)\n    return rows\n\n\ndef parse_twse_market(payload: dict, month: str) -> list[dict]:\n    # Must check explicit date and columns, not treat current snapshot as history.\n    month=_month(month)\n    if not isinstance(payload,dict) or str(payload.get('stat','')).upper() != 'OK':\n        raise SchemaBlocked('TWSE_MARKET_NOT_OK')\n    fields=payload.get('fields');data=payload.get('data')\n    if not isinstance(fields,list) or not isinstance(data,list) or not data:\n        raise SchemaBlocked('TWSE_MARKET_SCHEMA')\n    matches={}\n    for kind,word in [('open','開盤指數'),('high','最高指數'),('low','最低指數'),('close','收盤指數'),('date','日期')]:\n        hits=[i for i,f in enumerate(fields) if isinstance(f,str) and word in f]\n        if len(hits)!=1:\n            raise SchemaBlocked('TWSE_MARKET_COL:'+kind)\n        matches[kind]=hits[0]\n    out=[]\n    for row in data:\n        if not isinstance(row,list) or len(row)!=len(fields):\n            raise SchemaBlocked('TWSE_MARKET_ROW_SHAPE')\n        d=_date(row[matches['date']]);\n        if not d.startswith(month+'-'):\n            raise SchemaBlocked('TWSE_MARKET_WRONG_MONTH')\n        r={'market':'TWSE','index_id':'TAIEX','trade_date':d,\n           **{k:_number(row[matches[k]]) for k in ('open','high','low','close')}}\n        if not r['low']<=r['open']<=r['high'] or not r['low']<=r['close']<=r['high']:\n            raise SchemaBlocked('TWSE_MARKET_OHLC_INVALID')\n        out.append(r)\n    return _dedup(out,('trade_date','index_id'))\n\n\ndef parse_tpex_market(payload: dict, month: str) -> list[dict]:\n    month=_month(month)\n    tables=_table_objects(payload)\n    if not tables:\n        raise SchemaBlocked('TPEX_MARKET_UNKNOWN_SCHEMA')\n    out=[]\n    for title, rows in tables:\n        for row in rows:\n            d=_get_col(row,('資料日期','交易日期','日期','日　期'), contains=True)\n            close=_get_col(row,('收市','收盤指數','收盤價','收盤','收市指數'))\n            if d is None or close is None:\n                continue\n            try:\n                ds=_date(d)\n            except SchemaBlocked:\n                # Explicit table summary is not a tradable daily row.\n                continue\n            if not ds.startswith(month+'-'):\n                raise SchemaBlocked('TPEX_MARKET_WRONG_MONTH')\n            rec={'market':'TPEX','index_id':'TPEX','trade_date':ds,'close':_number(close)}\n            for k,labels in [('open',('開市','開盤指數','開盤')),('high',('最高','最高價','最高指數')),('low',('最低','最低價','最低指數'))]:\n                v=_get_col(row,labels)\n                if v is not None and str(v).strip() not in ('','--','－'):\n                    rec[k]=_number(v)\n            if all(k in rec for k in ('open','high','low')):\n                if not rec['low']<=rec['open']<=rec['high'] or not rec['low']<=rec['close']<=rec['high']:\n                    raise SchemaBlocked('TPEX_MARKET_OHLC_INVALID')\n            out.append(rec)\n    if not out:\n        raise SchemaBlocked('TPEX_MARKET_NO_VERIFIED_DAILY_ROWS')\n    return _dedup(out,('trade_date','index_id'))\n\n\ndef _is_sector(name: str) -> bool:\n    name=re.sub(r'\\s+','',name).replace('指數','')\n    if any(s in name for s in ('報酬','槓桿','反向','成交','臺指','台指','臺灣50','特選')):\n        return False\n    # Exclude market/cross-market aggregate indices and unverified generic names.\n    if any(s in name for s in ('加權股價','未含','寶島')):\n        return False\n    return ('類' in name or '業' in name) and len(name)<70\n\n\ndef parse_twse_sector(payload: dict, day: str) -> list[dict]:\n    day=_date(day)\n    if not isinstance(payload,dict):\n        raise SchemaBlocked('TWSE_SECTOR_OBJECT_REQUIRED')\n    pdate=payload.get('date') or payload.get('Date')\n    if pdate is None or _date(pdate)!=day:\n        raise SchemaBlocked('TWSE_SECTOR_DATE_NOT_VERIFIED')\n    tables=_table_objects(payload)\n    out=[]\n    for title, rows in tables:\n        for row in rows:\n            name=_get_col(row,('指數','指數名稱','類股指數','分類指數'))\n            close=_get_col(row,('收盤指數','收市指數','收盤'))\n            if name is None or close is None or not _is_sector(str(name)):\n                continue\n            out.append({'market':'TWSE','index_id':re.sub(r'\\s+','',str(name)),\n                        'trade_date':day,'close':_number(close),'type':'sector_price_index'})\n    if not out:\n        raise SchemaBlocked('TWSE_SECTOR_TABLE_NOT_CONFIRMED')\n    return _dedup(out,('trade_date','index_id'))\n\n\ndef parse_tpex_sector(payload: dict, month: str) -> list[dict]:\n    month=_month(month)\n    tables=_table_objects(payload)\n    if not tables:\n        raise SchemaBlocked('TPEX_SECTOR_UNKNOWN_SCHEMA')\n    out=[]\n    for title,rows in tables:\n        for row in rows:\n            d=_get_col(row,('日期','交易日','資料日期'),contains=True)\n            if d is None:\n                continue\n            try:\n                ds=_date(d)\n            except SchemaBlocked:\n                continue\n            if not ds.startswith(month+'-'):\n                raise SchemaBlocked('TPEX_SECTOR_WRONG_MONTH')\n            # Long rows (date, sector name, close), or monthly wide sector matrix.\n            name=_get_col(row,('產業分類','產業別','指數名稱','分類指數','類股指數'))\n            close=_get_col(row,('收盤指數','收盤','收市','收市指數'))\n            if name is not None and close is not None and _is_sector(str(name)):\n                out.append({'market':'TPEX','index_id':str(name).strip(),'trade_date':ds,\n                            'close':_number(close),'type':'sector_price_index'})\n            else:\n                for col,value in row.items():\n                    if _is_sector(str(col)) and str(value).strip() not in ('','--','－','—'):\n                        out.append({'market':'TPEX','index_id':re.sub(r'\\s+','',str(col)),\n                                    'trade_date':ds,'close':_number(value),'type':'sector_price_index'})\n    if not out:\n        raise SchemaBlocked('TPEX_SECTOR_NO_VERIFIED_DAILY_ROWS')\n    return _dedup(out,('trade_date','index_id'))\n\n\ndef url_for(source: str, stamp: str) -> str:\n    if source not in SOURCES:\n        raise ValueError('UNKNOWN_SOURCE')\n    if source == 'twse_sector':\n        ymd=_date(stamp).replace('-','')\n        args={'response':'json','date':ymd,'type':'ALL'}\n    else:\n        y,m=map(int,_month(stamp).split('-'))\n        if source=='twse_market':\n            args={'response':'json','date':f'{y:04d}{m:02d}01'}\n        else:\n            # TPEx historical reports use ROC month query; endpoint/schema must be\n            # checked by parse_tpex_* before the receipt can report VERIFIED.\n            args={'date':f'{y-1911}/{m:02d}/01','response':'json'}\n    return SOURCES[source]['endpoint']+'?'+urlencode(args)\n\n\ndef partition(base: Path, source: str, stamp: str) -> Path:\n    if source=='twse_sector':\n        stamp=_date(stamp)\n        return base/source/stamp[:7]/stamp\n    return base/source/_month(stamp)\n\n\ndef _verify_receipt(folder: Path) -> dict | None:\n    receipt=folder/'receipt.json'\n    if not receipt.exists():\n        return None\n    r=json.loads(receipt.read_text(encoding='utf-8'))\n    if r.get('status')!='VERIFIED':\n        return None\n    if r.get('version')!=VERSION:\n        raise ValueError('VERSION_MISMATCH_EXISTING_RECEIPT')\n    for filename,sha in r['checksums'].items():\n        file=folder/filename\n        if not file.is_file() or sha_bytes(file.read_bytes())!=sha:\n            raise ValueError('EXISTING_SUCCESS_HASH_MISMATCH:'+str(file))\n    return r\n\n\ndef _save_receipt(folder: Path, receipt: dict):\n    folder.mkdir(parents=True,exist_ok=True)\n    # Error receipt can be replaced by another attempt, verified is immutable.\n    if (folder/'receipt.json').exists():\n        previous=json.loads((folder/'receipt.json').read_text('utf-8'))\n        if previous.get('status')=='VERIFIED':\n            raise ValueError('ALREADY_VERIFIED_REFUSE_OVERWRITE')\n    write_new(folder/('receipt-last-attempt-'+receipt['attempt_id']+'.json'),json_bytes(receipt))\n    # Error status receipt is mutable pointer to preserved immutable attempts.\n    path=folder/'receipt.json'; tmp=folder/'receipt.tmp'\n    tmp.write_bytes(json_bytes(receipt));os.replace(tmp,path)\n\n\ndef write_partition(folder: Path, raw: bytes, normalized: list[dict], receipt: dict):\n    \"\"\"Raw first, normalized JSON always, and deterministic Parquet; no CURRENT edit.\"\"\"\n    import pandas as pd\n    # Colab has pyarrow installed in setup cell; fail rather than mark PASS w/o parquet.\n    import pyarrow  # noqa: F401\n    raw_sha=sha_bytes(raw)\n    raw_name='raw-'+raw_sha+'.bin'\n    write_new(folder/raw_name,raw)\n    normalized=_dedup(normalized,('trade_date','index_id'))\n    norm_name='normalized-'+sha_bytes(json_bytes(normalized))+'.json'\n    norm_sha=write_new(folder/norm_name,json_bytes(normalized))\n    df=pd.DataFrame(normalized)\n    buff=io.BytesIO();df.to_parquet(buff,index=False,engine='pyarrow')\n    pqraw=buff.getvalue();pq_name='index-'+sha_bytes(pqraw)+'.parquet'\n    pq_sha=write_new(folder/pq_name,pqraw)\n    probe=pd.read_parquet(folder/pq_name,engine='pyarrow')\n    if len(probe)!=len(normalized) or list(probe.columns)!=list(df.columns):\n        raise ValueError('PARQUET_READBACK_MISMATCH')\n    receipt.update({'status':'VERIFIED','rows':len(normalized),'first_date':normalized[0]['trade_date'],\n                    'last_date':normalized[-1]['trade_date'],\n                    'checksums':{raw_name:raw_sha,norm_name:norm_sha,pq_name:pq_sha},\n                    'parquet':pq_name,'normalized':norm_name})\n    _save_receipt(folder,receipt)\n\n\n@dataclass\nclass RunOptions:\n    start_month: str = '2025-01'\n    end_month: str | None = None\n    max_requests: int = 70\n    max_minutes: int = 15\n    interval_seconds: float = 2.5\n    include_twse_sector: bool = True\n    include_tpex_sector: bool = True\n    timeout_seconds: int = 30\n\n    def validate(self):\n        from datetime import timedelta\n        today=datetime.now(ZoneInfo('Asia/Taipei')).date()\n        last = (today.replace(day=1)-timedelta(days=1)).strftime('%Y-%m')\n        if self.end_month is None:self.end_month=last\n        if self.end_month>last:raise ValueError('CANNOT_COLLECT_CURRENT_OR_FUTURE_MONTH')\n        months_between(self.start_month,self.end_month)\n        if not 1<=self.max_requests<=300:raise ValueError('BAD_REQUEST_BUDGET')\n        if not 1<=self.max_minutes<=90:raise ValueError('BAD_MINUTE_BUDGET')\n        if not 2<=self.interval_seconds<=30:raise ValueError('RATE_LIMIT_INTERVAL_MUST_BE_AT_LEAST_2_SECONDS')\n        if not 5<=self.timeout_seconds<=60:raise ValueError('BAD_TIMEOUT')\n        return self\n\n\ndef _job_list(months: list[str],base: Path, *, include_twse_sector:bool, include_tpex_sector:bool):\n    \"\"\"Monthly index prioritized; daily TWSE sector uses verified trading dates.\"\"\"\n    for mo in months:\n        yield ('twse_market', mo)\n        yield ('tpex_market', mo)\n        if include_tpex_sector:\n            yield ('tpex_sector',mo)\n        if include_twse_sector:\n            market_folder=partition(base,'twse_market',mo)\n            verified=_verify_receipt(market_folder)\n            if verified:\n                for d in sorted(json.loads((market_folder/verified['normalized']).read_text('utf-8')),\n                                key=lambda x:x['trade_date']):\n                    yield ('twse_sector',d['trade_date'])\n\n\ndef backfill(root: str | Path, options: RunOptions, *, fetch=None, sleep=None, clock=None) -> dict:\n    \"\"\"Incremental official collection. The source endpoints are runtime-gated.\n\n    Cannot declare any target complete without actual normalized parquet + hash.\n    Source refusal stops that provider host, never switches identities/proxies.\n    \"\"\"\n    o=options.validate()\n    root=Path(root).expanduser().resolve()\n    if not root.exists():\n        raise FileNotFoundError('JANUS_DATASET_ROOT_DOES_NOT_EXIST:'+str(root))\n    out=(root/'derived'/'factor_inputs'/'official_index_backfill'/VERSION)\n    out.mkdir(parents=True,exist_ok=True)\n    months=months_between(o.start_month,o.end_month)\n    fetch=fetch or requests.get;sleep=sleep or time.sleep;clock=clock or time.monotonic\n    began=clock();calls=0;events=[];refused_hosts=set();invalid_sources=set();states={}\n    for source,stamp in _job_list(months,out,include_twse_sector=o.include_twse_sector,\n                                    include_tpex_sector=o.include_tpex_sector):\n        folder=partition(out,source,stamp)\n        past=_verify_receipt(folder)\n        if past:\n            states[source+'_SKIPPED_VERIFIED']=states.get(source+'_SKIPPED_VERIFIED',0)+1\n            continue\n        market=SOURCES[source]['market']\n        if source in invalid_sources:\n            events.append({'source':source,'period':stamp,'status':'SKIPPED_INVALID_SOURCE','rows':0,\n                           'reason':'SOURCE_PROBE_FAILED_EARLIER_IN_RUN'})\n            continue\n        if market in refused_hosts:\n            events.append({'source':source,'period':stamp,'status':'SOURCE_BLOCKED_HOST'});continue\n        if calls>=o.max_requests or (clock()-began)>o.max_minutes*60:\n            events.append({'source':source,'period':stamp,'status':'BUDGET_PAUSED'});break\n        url=url_for(source,stamp)\n        stamped=datetime.now(timezone.utc).isoformat()\n        try:\n            r=fetch(url,timeout=o.timeout_seconds,\n                    headers={'User-Agent':'JanusFourModelResearch/1.0; research only; manual Colab'})\n            calls+=1\n            status=int(r.status_code)\n            raw=r.content\n            if len(raw)>MAX_RAW_BYTES:\n                raise SchemaBlocked('SOURCE_RESPONSE_OVERSIZED')\n            folder.mkdir(parents=True,exist_ok=True)\n            rawsha=sha_bytes(raw)\n            write_new(folder/('raw-'+rawsha+'.bin'),raw)\n            receipt={'version':VERSION,'source':source,'market':market,'period':stamp,\n                'url':url,'source_page':SOURCES[source]['source_page'],\n                'source_rights_review_url':RIGHTS[source],\n                'source_endpoint_use_authorization':'REQUIRES_INDIVIDUAL_VERIFICATION',\n                'fetched_at_utc':stamped,'source_publication_time':'NOT_VERIFIED',\n                'historical_industry_membership_pit':'NOT_VERIFIED',\n                'http_status':status,'raw_sha256':rawsha,\n                'response_content_type':str((getattr(r,'headers',{}) or {}).get('Content-Type',''))[:128],\n                'publication_allowed':False,\n                'canonical_current_mutated':False,'status':'UNKNOWN',\n                'attempt_id':re.sub(r'[^0-9]','',stamped)[:20]+'-'+rawsha[:8]}\n            if status in BLOCK_STATUSES:\n                receipt['status']='SOURCE_BLOCKED'; refused_hosts.add(market)\n            elif status!=200:\n                receipt['status']='SOURCE_ENDPOINT_INVALID' if status in (404,410) else 'HTTP_ERROR'\n                if status in (404,410):\n                    invalid_sources.add(source)\n                    receipt['detail']='HTTP_SOURCE_PATH_NOT_FOUND'\n            else:\n                try:\n                    payload=json.loads(raw.decode('utf-8-sig'))\n                    parse={'twse_market':parse_twse_market,'tpex_market':parse_tpex_market,\n                           'twse_sector':parse_twse_sector,'tpex_sector':parse_tpex_sector}[source]\n                    normalized=parse(payload,stamp)\n                    if source in ('twse_market','tpex_market') and len(normalized)<10:\n                        raise SchemaBlocked('MONTH_HAS_FEWER_THAN_10_DAILY_INDEX_ROWS')\n                    if source=='twse_sector' and len(normalized)<5:\n                        raise SchemaBlocked('FEWER_THAN_5_TWSE_SECTOR_INDICES')\n                    if source=='tpex_sector':\n                        if len({r['trade_date'] for r in normalized})<10:\n                            raise SchemaBlocked('TPEX_SECTOR_MONTH_INCOMPLETE')\n                        if len({r['index_id'] for r in normalized})<5:\n                            raise SchemaBlocked('TPEX_SECTOR_TOO_FEW_INDUSTRIES')\n                    write_partition(folder,raw,normalized,receipt)\n                except (ValueError,KeyError,TypeError,UnicodeError) as exc:\n                    peek=raw[:4096].lstrip().lower()\n                    if peek.startswith((b'<!doctype html', b'<html')) or b'<html' in peek[:256]:\n                        title_404=b'<title>404' in peek or b'<h1>404' in peek\n                        receipt['status']='SOURCE_ENDPOINT_INVALID' if title_404 else 'SOURCE_FORMAT_UNSUPPORTED'\n                        receipt['detail']='HTTP_200_HTML_404' if title_404 else 'NON_JSON_HTML_PAGE_NOT_API'\n                        invalid_sources.add(source)\n                    else:\n                        receipt['status']='SCHEMA_BLOCKED';receipt['detail']=str(exc)[:240]\n            if receipt['status']!='VERIFIED':\n                _save_receipt(folder,receipt)\n            events.append({'source':source,'period':stamp,'status':receipt['status'],\n                           'rows':receipt.get('rows',0),'reason':receipt.get('detail','')})\n        except (requests.RequestException,OSError) as exc:\n            events.append({'source':source,'period':stamp,'status':'NETWORK_OR_IO_FAILED','reason':str(exc)[:160]})\n        if calls < o.max_requests and o.interval_seconds:\n            sleep(o.interval_seconds)\n    counts={}\n    for e in events:counts[e['status']]=counts.get(e['status'],0)+1\n    # Existing verified partitions are materialized, not just current-run statuses.\n    tally={}\n    for src in SOURCES:\n        total=0\n        if (out/src).exists():\n            for q in (out/src).rglob('receipt.json'):\n                if _verify_receipt(q.parent):total+=1\n        tally[src]=total\n    # Completeness is partition-level, never only requests_used or old 264 batches.\n    # Even a complete data capture does not certify historical sector membership/PIT.\n    pending=[]\n    for mo in months:\n        for src in ('twse_market','tpex_market') + (('tpex_sector',) if o.include_tpex_sector else ()):\n            if not _verify_receipt(partition(out,src,mo)):\n                pending.append({'source':src,'period':mo})\n        if o.include_twse_sector:\n            receipt=_verify_receipt(partition(out,'twse_market',mo))\n            if not receipt:\n                pending.append({'source':'twse_sector','period':mo,'reason':'TWSE_MONTH_CALENDAR_MISSING'})\n            else:\n                calendar=json.loads((partition(out,'twse_market',mo)/receipt['normalized']).read_text('utf-8'))\n                for day in sorted({r['trade_date'] for r in calendar}):\n                    if not _verify_receipt(partition(out,'twse_sector',day)):\n                        pending.append({'source':'twse_sector','period':day})\n    data_complete = len(pending)==0\n    summary={'version':VERSION,'created_at_utc':datetime.now(timezone.utc).isoformat(),\n        'start_month':o.start_month,'end_month':o.end_month,'requests_used':calls,\n        'request_budget':o.max_requests,'partition_verified_counts':tally,\n        'current_run_status_counts':counts,'status':('SOURCE_DATA_VERIFIED_COMPLETE' if data_complete else 'PARTIAL_OR_PENDING'),\n        'outstanding_partition_count':len(pending),'outstanding_first_20':pending[:20],\n        'source_authorization':'PER_SOURCE_REVIEW_REQUIRED',\n        'historical_industry_membership_pit':'NOT_VERIFIED',\n        'canonical_current_unchanged':True,'publication_allowed':False,\n        'reason':'僅以各月／日實際保存且雜湊通過者為 VERIFIED；未驗證來源不能宣稱全量完成。'}\n    reports=root/'reports';reports.mkdir(exist_ok=True)\n    # Keep distinct dated run receipt + latest convenience files.\n    runid=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')\n    write_new(reports/('大盤產業補收_'+runid+'.json'),json_bytes({'summary':summary,'events':events}))\n    (reports/'大盤產業補收_latest.json').write_bytes(json_bytes({'summary':summary,'events':events}))\n    with (reports/'大盤產業補收_latest.csv').open('w',newline='',encoding='utf-8-sig') as handle:\n        w=csv.DictWriter(handle,fieldnames=('source','period','status','rows','reason'));w.writeheader()\n        for e in events:w.writerow({k:e.get(k,'') for k in w.fieldnames})\n    summary['reports_dir']=str(reports)\n    summary['research_data_dir']=str(out)\n    return summary\n"
from pathlib import Path
import sys, json, importlib
CODE_DIR = Path('/content/janus_index_collector_v384')
CODE_DIR.mkdir(parents=True, exist_ok=True)
(CODE_DIR/'official_index_backfill.py').write_text(BACKFILL_CODE,encoding='utf-8')
if str(CODE_DIR) not in sys.path: sys.path.insert(0,str(CODE_DIR))
sys.modules.pop('official_index_backfill', None)
from official_index_backfill import RunOptions, backfill
cfg=RunOptions(start_month=START_MONTH,end_month=END_MONTH,max_requests=REQUEST_BUDGET,
               max_minutes=MAX_MINUTES,interval_seconds=REQUEST_INTERVAL_SECONDS,
               include_twse_sector=INCLUDE_TWSE_SECTOR,include_tpex_sector=INCLUDE_TPEX_SECTOR)
summary=backfill(ROOT,cfg)
print(json.dumps(summary,ensure_ascii=False,indent=2))


## 4．檢視報告


In [ ]:
from pathlib import Path
import pandas as pd
p=ROOT/'reports'/'大盤產業補收_latest.csv'
if p.is_file():
    report=pd.read_csv(p)
    print(report.groupby(['source','status'],dropna=False).size().to_string())
    display(report.head(30))
else:
    print('尚未有本輪報告')


## 5．驗收界線與後續

TWSE 大盤改用歷史報表路徑但仍待 Colab 真實 JSON 確認；TPEx 官方頁面不等於已確認的 JSON API，遇 HTML 時 fail closed，絕不視為 VERIFIED。只有日期、筆數、Parquet SHA、讀回通過才能標記成功。TPEx 產業月報延後更新，實際發布時點和個股歷史產業歸屬 PIT 仍需獨立驗證。報告保存在 reports/，研究分區放 derived/factor_inputs/official_index_backfill/v3.8.4/。
